# Qwen3-0.6B × 커스텀 Triton AttentionBackend on vLLM

이 노트북은 vLLM 의 공식 `AttentionBackend` 스펙을 서브클래싱하고,
**Python entry point (`vllm.general_plugins`) 로 자동 등록** 하는 정식 플러그인 경로를 보여준다.

**범위 고지**
- prefill / decode 를 **각각 전용 Triton 커널**로 구현 (`_fwd_kernel_prefill`, `_fwd_kernel_decode`)
- 커널은 dtype(fp16/bf16) · head\_dim(2의 거듭제곱) generic. 이 노트북은 Qwen3-0.6B fp16 기준으로 실행
- `max_num_seqs=1`, `enforce_eager=True` 단일 시퀀스 한정 (plugin 덕분에 subprocess 경계는 OK)


## 준비물 & 설치

- CUDA GPU 필수
- Python >= 3.10
- vLLM **0.19.1 정확히** 권장 (내부 API drift)

```bash
# 노트북 디렉토리에서
pip install -e .
```

`pyproject.toml` 의 entry point (`vllm.general_plugins = triton_attention_backend:register`) 가
모든 vLLM 프로세스(메인·엔진 코어·워커) 에서 자동으로 `register()` 를 호출해 준다.
즉 **노트북 안에서 `register()` 를 수동 호출하지 않는다**.

절차:
1. cell 3 — 환경 확인
2. cell 8 — 커널 smoke test
3. cell 10 — LLM 생성 (plugin 이 이미 CUSTOM 슬롯 점유)
4. cell 11~13 — generate + 실행 증거 확인


In [ ]:
import torch, triton, vllm

print('cuda:', torch.cuda.is_available())
print('gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
print('vllm:', vllm.__version__)
print('triton:', triton.__version__)

if not torch.cuda.is_available():
    raise SystemExit('이 노트북은 CUDA GPU가 필요합니다.')

# 이 노트북은 vLLM 0.19.x 기준으로 작성됨. 다른 버전에서는 내부 API drift 가능.
assert vllm.__version__.startswith('0.19'), (
    f'vLLM 0.19.x 권장 (현재: {vllm.__version__}). '
    '다른 버전은 AttentionBackend 내부 API 가 다를 수 있음.'
)


## Qwen3-0.6B 구조 요약

| 항목 | 값 |
|---|---|
| hidden_size | 1024 |
| Q heads | 16 |
| KV heads | 8 (GQA 2:1) |
| head_dim | 128 |
| layers | 28 |

In [ ]:
from transformers import AutoConfig

cfg = AutoConfig.from_pretrained('Qwen/Qwen3-0.6B')
hd = cfg.head_dim if hasattr(cfg, 'head_dim') else cfg.hidden_size // cfg.num_attention_heads
print('hidden:', cfg.hidden_size)
print('Q heads:', cfg.num_attention_heads, '/ KV heads:', cfg.num_key_value_heads)
print('head_dim:', hd)
print('layers:', cfg.num_hidden_layers)

## Triton 커널 역할

`triton_attn.py` 에는 두 개의 `@triton.jit` 커널이 있다:

- **`_fwd_kernel_prefill`** — q\_len == kv\_len (causal). `triton_attention_prefill(q, k, v)` 래퍼에서 호출.
- **`_fwd_kernel_decode`** — q\_len == 1, kv\_len >= 1, causal mask 없음 (마지막 쿼리는 모든 과거 key 를 봄).
  `triton_attention_decode(q, k, v)` 래퍼에서 호출. 비용 O(s\_len).

둘 다 Flash-style online softmax, fp16/bf16 generic, head\_dim 2의 거듭제곱.
Backend (`MyTritonImpl.forward`) 는 `q_len == s_len` 분기로 둘 중 하나를 호출한다.


In [ ]:
from triton_attn import triton_attention

print(triton_attention.__doc__)

In [ ]:
# 커널 smoke test: 랜덤 Q/K/V → SDPA vs ours → max_err 확인
import torch.nn.functional as F

B, Hq, Hkv, S, D = 1, 16, 8, 128, 128
q = torch.randn(B, Hq, S, D, dtype=torch.float16, device='cuda')
k = torch.randn(B, Hkv, S, D, dtype=torch.float16, device='cuda')
v = torch.randn(B, Hkv, S, D, dtype=torch.float16, device='cuda')

# SDPA 기준값 (GQA expand 포함)
ref = F.scaled_dot_product_attention(
    q, k.repeat_interleave(Hq // Hkv, dim=1),
    v.repeat_interleave(Hq // Hkv, dim=1), is_causal=True
)
ours = triton_attention(q, k, v)

max_err = (ours.float() - ref.float()).abs().max().item()
print(f'max_abs_err = {max_err:.6f}  →  ', 'PASS' if max_err < 1e-2 else 'FAIL')

## Plugin entry point 로 자동 등록

`pyproject.toml`:
```toml
[project.entry-points."vllm.general_plugins"]
my_triton_backend = "triton_attention_backend:register"
```

`pip install -e .` 이후 vLLM 은 메인 · 엔진 코어 · 워커 프로세스 모두에서
`register()` 를 자동 호출해 `AttentionBackendEnum.CUSTOM` 슬롯에 `MyTritonBackend` 를 꽂는다.

실행 증거는 엔진 코어 stderr 에 찍히는 `MyTritonImpl.forward fired ...` 로그로 확인한다
(`FIRE_COUNTER` 는 프로세스별 독립 메모리라 메인에서 읽으면 0 일 수 있음).


In [ ]:
# entry point 등록 상태 확인 (pip install -e . 이 성공했다면 등록되어 있어야 함)
from importlib.metadata import entry_points

eps = list(entry_points(group='vllm.general_plugins'))
mine = [e for e in eps if e.name == 'my_triton_backend']
assert mine, (
    '`my_triton_backend` entry point 가 보이지 않는다. '
    '노트북 디렉토리에서 `pip install -e .` 을 실행했는지 확인하라.'
)
print('plugin registered:', mine[0].name, '->', mine[0].value)


In [ ]:
from vllm import LLM, SamplingParams
from vllm.v1.attention.backends.registry import AttentionBackendEnum

llm = LLM(
    model='Qwen/Qwen3-0.6B',
    dtype='float16',
    attention_backend=AttentionBackendEnum.CUSTOM,
    enforce_eager=True,
    max_num_seqs=1,
    max_model_len=2048,
)


In [ ]:
out = llm.generate(
    ['The capital of France is'],
    SamplingParams(temperature=0, max_tokens=32)
)
print(out[0].outputs[0].text)


## 검증: "실제로 우리 백엔드가 실행됐는가"

plugin 방식에서는 엔진 코어가 **별도 프로세스**에서 돌기 때문에
메모리 내 `FIRE_COUNTER` 는 이 노트북 프로세스에 전달되지 않는다 (읽으면 0).

대신 위 `llm.generate(...)` 출력 **바로 위** 에 엔진 코어 stderr 가 다음과 같이 찍혔어야 한다:

```
(EngineCore pid=XXXXX) INFO ... [cuda.py] Using AttentionBackendEnum.CUSTOM backend.
(EngineCore pid=XXXXX) WARNING ... MyTritonImpl.forward fired (prefill) tokens=5
(EngineCore pid=XXXXX) WARNING ... MyTritonImpl.forward fired (decode) s_len=6
```

두 `fired` 로그가 모두 찍혔으면 prefill/decode 양쪽에서 Triton 커널이 실제로 호출된 증거다.
(로그는 각 경로 첫 1 회만 기록 — `MY_TRITON_BACKEND_LOG=0` 으로 끌 수도 있음)


In [ ]:
# 이 노트북 프로세스에서 확인 가능한 것:
# (1) plugin 이 등록돼 있고
# (2) CUSTOM 슬롯에 우리 백엔드 경로가 꽂혔는가
from vllm.v1.attention.backends.registry import AttentionBackendEnum

path = AttentionBackendEnum.CUSTOM.get_path()
print('CUSTOM slot ->', path)
assert 'MyTritonBackend' in path, 'CUSTOM slot 에 우리 백엔드가 등록되지 않음'
print('OK — 노트북 프로세스에서도 plugin 자동 등록 확인')
print()
print('실제 커널 실행 증거는 위 cell 11 출력 바로 위의 `MyTritonImpl.forward fired ...` 로그를 확인할 것.')


## 다음 단계

- **decode 를 split-k (stage1/stage2) 로 최적화** — 긴 컨텍스트에서 parallelism 확보
  (vLLM `triton_decode_attention.py` 의 `_fwd_kernel_stage1/2` 패턴 참고)
- **fp8 KV cache**: `supported_kv_cache_dtypes` 에 fp8 추가 + 커널의 `reshape_and_cache` 경로 조정
- **unified (prefill + decode 혼합 배치)**: per-sequence `query_start_loc` / `seq_lens` 기반 단일 커널로 수렴
